In [ ]:
# MATLAB section 1/6 for HybridFilterExample: Hybrid Point Process Filter Example

# % Hybrid Point Process Filter Example
# This example is based on an implementation of the Hybrid Point Process
# filter described in _General-purpose filter design for neural prosthetic
# devices_ by Srinivasan L, Eden UT, Mitter SK, Brown EN in J Neurophysiol.
# 2007 Oct, 98(4):2456-75.
#
#

# Python translation bootstrap for this helpfile.

# Topic: HybridFilterExample
# Execution group: full
# Workflow family: network
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/HybridFilterExample.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HybridFilterExample"
FAMILY = "network"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HybridFilterExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HybridFilterExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HybridFilterExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HybridFilterExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 3

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)


In [ ]:
# MATLAB section 2/6 for HybridFilterExample: Problem Statement

# % Problem Statement
# Suppose that a process of interest can be modeled as consisting of
# several discrete states where the evolution of the system under each
# state can be modeled as a linear state space model. The observations of
# both the state and the continuous dynamics are not direct, but rather
# observed through how the continuous and discrete states affect the firing
# of a population of neurons. The goal of the hybrid filter is to estimate
# both the continuous dynamics and the underlying system state from only
# the neural population firing (point process observations).
#
# To illustrate the use of this filter, we consider a reaching task. We
# assume two underlying system states s=1="Not Moving"=NM and s=2="Moving"=M.
# Under the "Not Moving" the position of the arm remain constant,
# whereas in the "Moving" state, the position and velocities evolved based
# on the arm acceleration that is modeled as a gaussian white noise
# process.
#
# Under both the "Moving" and "Not Moving" states, the arm evolution state
# vector is

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/6 for HybridFilterExample: Section

# %
#
# $${\bf{x}} = {[x,y,{v_x},{v_y},{a_x},{a_y}]^T}$$
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/6 for HybridFilterExample: Generated Simulated Arm Reach

# % Generated Simulated Arm Reach
#
# MATLAB: clear all;
# MATLAB: close all;
# MATLAB: delta=0.001;
# MATLAB: Tmax=2;
# MATLAB: time=0:delta:Tmax;
# MATLAB: A{2} = [1 0 delta 0     delta^2/2   0;
# MATLAB:         0 1 0     delta 0           delta^2/2;
# MATLAB:         0 0 1     0     delta       0;
# MATLAB:         0 0 0     1     0           delta;
# MATLAB:         0 0 0     0     1           0;
# MATLAB:         0 0 0     0     0           1];
#
# MATLAB: A{1} = [1 0 0 0 0 0;
# MATLAB:         0 1 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0;
# MATLAB:         0 0 0 0 0 0];
# MATLAB: A{1} = [1 0;
# MATLAB:         0 1];
#
# MATLAB: Px0{2} =1e-6*eye(6,6);
# MATLAB: Px0{1} =1e-6*eye(2,2);
#
# MATLAB: minCovVal = 1e-12;
# MATLAB: covVal = 1e-3;
#
#
#
# MATLAB: Q{2}=[minCovVal     0   0     0   0       0;
# MATLAB:       0       minCovVal 0     0   0       0;
# MATLAB:       0       0   minCovVal   0   0       0;
# MATLAB:       0       0   0     minCovVal 0       0;
# MATLAB:       0       0   0     0   covVal      0;
# MATLAB:       0       0   0     0   0       covVal];
#
# MATLAB: Q{1}=minCovVal*eye(2,2);
#
# MATLAB: mstate = zeros(1,length(time));
# MATLAB: ind{1}=1:2;
# MATLAB: ind{2}=1:6;
#
# Acceleration model
# MATLAB: X=zeros(max([size(A{1},1),size(A{2},1)]),length(time));
# MATLAB: p_ij = [.998 .002;
# MATLAB:         .001 .999];
#
# MATLAB: for i = 1:length(time)
#
# MATLAB:     if(i==1)
# MATLAB:         mstate(i) = 1;
# MATLAB:     else
# MATLAB:        if(rand(1,1)<p_ij(mstate(i-1),mstate(i-1)))
# MATLAB:             mstate(i) = mstate(i-1);
# MATLAB:        else
# MATLAB:            if(mstate(i-1)==1)
# MATLAB:                mstate(i) = 2;
# MATLAB:            else
# MATLAB:                mstate(i) = 1;
# MATLAB:            end
# MATLAB:        end
# MATLAB:     end
# MATLAB:     st = mstate(i);
# MATLAB:     R=chol(Q{st});
# MATLAB:     if(i<length(time))
# MATLAB:         X(ind{st},i+1) = A{st}*X(ind{st},i) + R*randn(length(ind{st}),1);
# MATLAB:     end
#
# MATLAB: end

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/6 for HybridFilterExample: Section

# %
# save paperHybridFilterExample time Tmax delta mstate X p_ij ind A Q Px0
# MATLAB: load paperHybridFilterExample;
# MATLAB: Q{1}=minCovVal*eye(2,2);
# MATLAB: numCells=40;
# MATLAB: close all;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:     scrsz(3)*.8 scrsz(4)*.9]);
# MATLAB: subplot(4,2,[1 3]);
# MATLAB: plot(100*X(1,:),100*X(2,:),'k','Linewidth',2); hx=xlabel('X [cm]');
# MATLAB: hy=ylabel('Y [cm]');  hold on;
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: title('Reach Path','FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB: hold on;
# MATLAB: h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',16);
# MATLAB: h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',16);
# MATLAB: legend([h1 h2],'Start','Finish','Location','NorthEast');
#
#
#
# MATLAB: subplot(4,2,[6 8]);
# MATLAB: plot(time,mstate,'k','Linewidth',2); axis tight; v=axis;
# MATLAB: axis([v(1) v(2) 0 3]);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('state');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: set(gca,'YTick',[1 2],'YTickLabel',{'N','M'})
# MATLAB: title('Discrete Movement State','FontWeight','bold','Fontsize',14,...
# MATLAB:     'FontName','Arial');
#
# MATLAB: subplot(4,2,5);
# MATLAB: h1=plot(time,100*X(1,1:end),'k','Linewidth',2); hold on;
# MATLAB: h2=plot(time,100*X(2,1:end),'k-.','Linewidth',2);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('Position [cm]');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: h_legend=legend([h1,h2],'x','y','Location','NorthEast');
# MATLAB: set(h_legend,'FontSize',14)
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
#
#
# MATLAB: subplot(4,2,7);
# MATLAB: h1=plot(time,100*X(3,1:end),'k','Linewidth',2); hold on;
# MATLAB: h2=plot(time,100*X(4,1:end),'k-.','Linewidth',2);
# MATLAB: hx=xlabel('time [s]'); hy=ylabel('Velocity [cm/s]');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
# MATLAB: h_legend=legend([h1,h2],'v_{x}','v_{y}','Location','NorthEast');
# MATLAB: set(h_legend,'FontSize',14)
# MATLAB: pos = get(h_legend,'position');
# MATLAB: set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);
#
# MATLAB: meanMu = log(10*delta);  % baseline firing rate
# MATLAB: MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
# MATLAB: coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...
# MATLAB:     0*randn(numCells,2)];
# Add realization by thinning with history
# MATLAB: dataMat = [ones(size(X,2),1),X(:,1:end)'];
# Generate M1 cells
# MATLAB: clear lambda tempSpikeColl lambdaCIF n;
# MATLAB: fitType ='binomial';
# matlabpool open;
# MATLAB:  for i=1:numCells
# MATLAB:      tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:      if(strcmp(fitType,'binomial'));
# MATLAB:         lambdaData = tempData./(1+tempData);
# MATLAB:      else
# MATLAB:         lambdaData = tempData;
# MATLAB:      end
# MATLAB:      lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:          '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:          {strcat('\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:      maxTimeRes = 0.001;
# MATLAB:      tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);
# MATLAB:      n{i} = tempSpikeColl{i}.getNST(1);
# MATLAB:      n{i}.setName(num2str(i));
# MATLAB:  end
# MATLAB:  spikeColl = nstColl(n);
# MATLAB:  subplot(4,2,[2 4]);
# MATLAB: spikeColl.plot;
# MATLAB: set(gca,'xtick',[],'xtickLabel',[],'ytickLabel',[]);
# MATLAB: title('Neural Raster','FontWeight','bold','Fontsize',14,'FontName','Arial');
# MATLAB: hx=xlabel('time [s]','Interpreter','none');
# MATLAB: hy=ylabel('Cell Number','Interpreter','none');
# MATLAB: set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');
#
# close all;
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
fig = FIGURE_TRACKER.new_figure(section_index=5, matlab_line_number=110, matlab_snippet="fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...")
ref_image = (MATLAB_HELP_ROOT / "HybridFilterExample.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 5, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/6 for HybridFilterExample: Simulate Neural Firing

# % Simulate Neural Firing
# We simulate a population of neurons that fire in response to the movement
# velocity (x and y coorinates)
#
# Use the data to estimate the process noise for the moving case and
# non-moving case
#
# MATLAB: nonMovingInd = intersect(find(X(5,:)==0),find(X(6,:)==0));
# MATLAB: movingInd=setdiff(1:size(X,2),nonMovingInd);
# MATLAB: Q{2}=diag(var(diff(X(:,movingInd),[],2),[],2));
# MATLAB: Q{2}(1:4,1:4)=0;
# MATLAB: varNV=diag(var(diff(X(:,nonMovingInd),[],2),[],2));
# MATLAB: Q{1} = varNV(1:2,1:2);
# MATLAB: close all; clear S_est X_est MU_est S_estNT X_estNT MU_estNT;
# MATLAB: numExamples = 20;
# MATLAB: numCells=40;
# MATLAB: scrsz = get(0,'ScreenSize');
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
# MATLAB:     scrsz(3)*.9 scrsz(4)*.9]);
#
# MATLAB: for n=1:numExamples
# MATLAB:     meanMu = log(10*delta);  % baseline firing rate
# MATLAB:     MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)
# MATLAB:     coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...
# MATLAB:         0*randn(numCells,2)];
#
#
#
# Add realization by thinning with history
# MATLAB:     dataMat = [ones(size(X,2),1),X(:,1:end)'];
# Generate M1 cells
# MATLAB:     clear lambda tempSpikeColl lambdaCIF nst;
# MATLAB:     fitType ='binomial';
# matlabpool open;
# MATLAB:      for i=1:numCells
# MATLAB:          tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:          if(strcmp(fitType,'binomial'));
# MATLAB:             lambdaData = tempData./(1+tempData);
# MATLAB:          else
# MATLAB:             lambdaData = tempData;
# MATLAB:          end
# MATLAB:          lambda{i}=Covariate(time,lambdaData./delta, ...
# MATLAB:              '\Lambda(t)','time','s','spikes/sec',...
# MATLAB:              {strcat('\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});
# MATLAB:          maxTimeRes = 0.001;
# MATLAB:          tempSpikeColl{i} = ...
# MATLAB:              CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);
# MATLAB:          nst{i} = tempSpikeColl{i}.getNST(1);
# MATLAB:          nst{i}.setName(num2str(i));
# MATLAB:      end
#
# Decode the x-y trajectory
#
# Enforce that the maximum time resolution is delta
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     spikeColl.resample(1/delta);
# MATLAB:     dN = spikeColl.dataToMatrix;
# MATLAB:     dN(dN>1)=1; %Avoid more than 1 spike per bin.
#
# Starting states are equally probable
# MATLAB:     Mu0=.5*ones(size(p_ij,1),1);
# MATLAB:     clear x0 yT clear Pi0 PiT;
# MATLAB:     x0{1} = X(ind{1},1);
# MATLAB:     yT{1} = X(ind{1},end);
# MATLAB:     Pi0    = Px0;
# MATLAB:     PiT{1} = 1e-9*eye(size(x0{1},1),size(x0{1},1));
#
# MATLAB:     x0{2} = X(ind{2},1);
# MATLAB:     yT{2} = X(ind{2},end);
# MATLAB:     PiT{2} = 1e-9*eye(size(x0{2},1),size(x0{2},1));
#
#
# Run the Hybrid Point Process Filter
# MATLAB:     [S_est, X_est, W_est, MU_est, X_s, W_s,pNGivenS]=...
# MATLAB:         DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...
# MATLAB:         coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0,Pi0, yT,PiT);
# MATLAB:     [S_estNT, X_estNT, W_estNT, MU_estNT, X_sNT, W_sNT,pNGivenSNT]=...
# MATLAB:         DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...
# MATLAB:         coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0);
#
# Store the results for computing relevant statistics later
# MATLAB:     X_estAll(:,:,n) = X_est;
# MATLAB:     X_estNTAll(:,:,n) = X_estNT;
# MATLAB:     S_estAll(n,:)=S_est;
# MATLAB:     S_estNTAll(n,:)=S_estNT;
# MATLAB:     MU_estAll(:,:,n)=MU_est;
# MATLAB:     MU_estNTAll(:,:,n) = MU_estNT;
#
#
# State Estimate
# MATLAB:     subplot(4,3,[1 4]);
# MATLAB:     plot(time,mstate,'k','LineWidth',3); hold all;
# MATLAB:     plot(time,S_est,'b-.','Linewidth',.5);
# MATLAB:     plot(time,S_estNT,'g-.','Linewidth',.5); axis tight; v=axis;
# MATLAB:     axis([v(1) v(2) 0.5 2.5]);
#
# Movement State Probability (Non-movement State probability is 1-Pr(Movement))
# MATLAB:     subplot(4,3,[7 10]);
# MATLAB:     plot(time,MU_est(2,:),'b-.','Linewidth',.5);  hold on;
# MATLAB:     plot(time,MU_estNT(2,:),'g-.','Linewidth',.5);  hold on;
# MATLAB:     axis([min(time) max(time) 0 1.1]);
#
# The movement path
# MATLAB:     subplot(4,3,[2 3 5 6]);
# MATLAB:     h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;
# MATLAB:     h2=plot(100*X_est(1,:)',100*X_est(2,:)','b-.'); hold all;
# MATLAB:     h3=plot(100*X_estNT(1,:)',100*X_estNT(2,:)','g-.');
#
# X-Position
# MATLAB:     subplot(4,3,8);
# MATLAB:     h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(1,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(1,:)','g-.');
#
# Y-Position
# MATLAB:     subplot(4,3,9);
# MATLAB:     h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(2,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(2,:)','g-.');
#
# X-Velocity
# MATLAB:     subplot(4,3,11);
# MATLAB:     h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(3,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(3,:)','g-.');
#
# MATLAB:     subplot(4,3,12);
# MATLAB:     h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,100*X_est(4,:)','b-.');
# MATLAB:     h3=plot(time,100*X_estNT(4,:)','g-.');
#
#
#
#
# MATLAB: end
#
#
# Save all the example Data
# save Experiment6ReachExamples X_estAll X_estNTAll S_estAll ...
# S_estNTAll MU_estAll MU_estNTAll;
#
# load Experiment6ReachExamples;
#
# Mean Discrete State Estimate
# MATLAB:     subplot(4,3,[1 4]);
# MATLAB:     hold all;
# MATLAB:     plot(time,mstate,'k','LineWidth',3);
# MATLAB:     plot(time,mean(S_estAll),'b','LineWidth',3);
# MATLAB:     plot(time,mean(S_estNTAll),'g','LineWidth',3);
# MATLAB:     set(gca,'xtick',[],'YTick',[1 2.1],'YTickLabel',{'N','M'});
# MATLAB:     hy=ylabel('state'); hx=xlabel('time [s]');
# MATLAB:     set([hy hx],'FontName', 'Arial','FontSize',10,'FontWeight','bold',...
# MATLAB:         'Interpreter','none');
# MATLAB:     title('Estimated vs. Actual State','FontWeight','bold','Fontsize',...
# MATLAB:         12,'FontName','Arial');
#
#
#
#
# Mean State Movement State Probability
# MATLAB:     subplot(4,3,[7 10]);
# MATLAB:     plot(time, mean(squeeze(MU_estAll(2,:,:)),2),'b','LineWidth',3);
# MATLAB:     hold on;
# MATLAB:     plot(time,mean(squeeze(MU_estNTAll(2,:,:)),2),'g','LineWidth',3);
# MATLAB:     hold on;
# MATLAB:     axis([min(time) max(time) 0 1.1]);
# MATLAB:     hx=xlabel('time [s]'); hy=ylabel('P(s(t)=M | data)');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Probability of State','FontWeight','bold','Fontsize',12,...
# MATLAB:         'FontName','Arial');
#
# Mean movement path
# MATLAB:     subplot(4,3,[2 3 5 6]);
# MATLAB:     h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;
# MATLAB:     mXestAll=mean(100*X_estAll,3);
# MATLAB:     mXestNTAll=mean(100*X_estNTAll,3);
# MATLAB:     plot(mXestAll(1,:),mXestAll(2,:),'b','Linewidth',3);
# MATLAB:     plot(mXestNTAll(1,:),mXestNTAll(2,:),'g','Linewidth',3);
# MATLAB:     hx=xlabel('x [cm]'); hy=ylabel('y [cm]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
#
# MATLAB:     h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',14); hold on;
# MATLAB:     h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',14);
# MATLAB:     legend([h1 h2],'Start','Finish','Location','NorthEast');
# MATLAB:     title('Estimated vs. Actual Reach Path','FontWeight','bold',...
# MATLAB:         'Fontsize',12,'FontName','Arial');
#
#
# Mean X-Positon
# MATLAB:     subplot(4,3,8);
# MATLAB:     h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(1,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(1,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('x(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# Mean Y-Position
# MATLAB:     subplot(4,3,9);
# MATLAB:     h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(2,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(2,:),'g','LineWidth',3); hold on;
# MATLAB:     h_legend=legend([h1(1) h2(1) h3(1)],'Actual','PPAF+Goal',...
# MATLAB:         'PPAF','Location','SouthEast');
# MATLAB:     hy=ylabel('y(t) [cm]'); hx=xlabel('time [s]');
# MATLAB:     set(gca,'xtick',[],'xtickLabel',[]);
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Position','FontWeight','bold','Fontsize',12,'FontName','Arial');
# MATLAB:     set(h_legend,'FontSize',10)
# MATLAB:     pos = get(h_legend,'position');
# MATLAB:     set(h_legend, 'position',[pos(1)-.40 pos(2)+.51 pos(3:4)]);
#
# Mean X-Velocity
# MATLAB:     subplot(4,3,11);
# MATLAB:     h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(3,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(3,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('v_{x}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('X Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');
#
# Mean Y-Velocity
# MATLAB:     subplot(4,3,12);
# MATLAB:     h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;
# MATLAB:     h2=plot(time,mXestAll(4,:),'b','LineWidth',3); hold on;
# MATLAB:     h3=plot(time,mXestNTAll(4,:),'g','LineWidth',3); hold on;
# MATLAB:     hy=ylabel('v_{y}(t) [cm/s]'); hx=xlabel('time [s]');
# MATLAB:     set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');
# MATLAB:     title('Y Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=207, matlab_snippet="fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...")
ref_image = (MATLAB_HELP_ROOT / "HybridFilterExample_01.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 6, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #3 for HybridFilterExample>
fig = FIGURE_TRACKER.new_figure(section_index=6, matlab_line_number=190, matlab_snippet="<synthetic MATLAB figure event #3 for HybridFilterExample>")
ref_image = (MATLAB_HELP_ROOT / "HybridFilterExample_02.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 6, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "delta=0.001;",
    "Tmax=2;",
    "time=0:delta:Tmax;",
    "A{2} = [1 0 delta 0     delta^2/2   0;",
    "0 1 0     delta 0           delta^2/2;",
    "0 0 1     0     delta       0;",
    "0 0 0     1     0           delta;",
    "0 0 0     0     1           0;",
    "0 0 0     0     0           1];",
    "A{1} = [1 0 0 0 0 0;",
    "0 1 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0];",
    "A{1} = [1 0;",
    "0 1];",
    "Px0{2} =1e-6*eye(6,6);",
    "Px0{1} =1e-6*eye(2,2);",
    "minCovVal = 1e-12;",
    "covVal = 1e-3;",
    "Q{2}=[minCovVal     0   0     0   0       0;",
    "0       minCovVal 0     0   0       0;",
    "0       0   minCovVal   0   0       0;",
    "0       0   0     minCovVal 0       0;",
    "0       0   0     0   covVal      0;",
    "0       0   0     0   0       covVal];",
    "Q{1}=minCovVal*eye(2,2);",
    "mstate = zeros(1,length(time));",
    "ind{1}=1:2;",
    "ind{2}=1:6;",
    "X=zeros(max([size(A{1},1),size(A{2},1)]),length(time));",
    "p_ij = [.998 .002;",
    ".001 .999];",
    "for i = 1:length(time)",
    "if(i==1)",
    "mstate(i) = 1;",
    "else",
    "if(rand(1,1)<p_ij(mstate(i-1),mstate(i-1)))",
    "mstate(i) = mstate(i-1);",
    "else",
    "if(mstate(i-1)==1)",
    "mstate(i) = 2;",
    "else",
    "mstate(i) = 1;",
    "end",
    "end",
    "end",
    "st = mstate(i);",
    "R=chol(Q{st});",
    "if(i<length(time))",
    "X(ind{st},i+1) = A{st}*X(ind{st},i) + R*randn(length(ind{st}),1);",
    "end",
    "end",
    "load paperHybridFilterExample;",
    "Q{1}=minCovVal*eye(2,2);",
    "numCells=40;",
    "close all;",
    "scrsz = get(0,'ScreenSize');",
    "fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...",
    "scrsz(3)*.8 scrsz(4)*.9]);",
    "subplot(4,2,[1 3]);",
    "plot(100*X(1,:),100*X(2,:),'k','Linewidth',2); hx=xlabel('X [cm]');",
    "hy=ylabel('Y [cm]');  hold on;",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "title('Reach Path','FontWeight','bold','Fontsize',14,'FontName','Arial');",
    "hold on;",
    "h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',16);",
    "h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',16);",
    "legend([h1 h2],'Start','Finish','Location','NorthEast');",
    "subplot(4,2,[6 8]);",
    "plot(time,mstate,'k','Linewidth',2); axis tight; v=axis;",
    "axis([v(1) v(2) 0 3]);",
    "hx=xlabel('time [s]'); hy=ylabel('state');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "set(gca,'YTick',[1 2],'YTickLabel',{'N','M'})",
    "title('Discrete Movement State','FontWeight','bold','Fontsize',14,...",
    "'FontName','Arial');",
    "subplot(4,2,5);",
    "h1=plot(time,100*X(1,1:end),'k','Linewidth',2); hold on;",
    "h2=plot(time,100*X(2,1:end),'k-.','Linewidth',2);",
    "hx=xlabel('time [s]'); hy=ylabel('Position [cm]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "h_legend=legend([h1,h2],'x','y','Location','NorthEast');",
    "set(h_legend,'FontSize',14)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);",
    "subplot(4,2,7);",
    "h1=plot(time,100*X(3,1:end),'k','Linewidth',2); hold on;",
    "h2=plot(time,100*X(4,1:end),'k-.','Linewidth',2);",
    "hx=xlabel('time [s]'); hy=ylabel('Velocity [cm/s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "h_legend=legend([h1,h2],'v_{x}','v_{y}','Location','NorthEast');",
    "set(h_legend,'FontSize',14)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);",
    "meanMu = log(10*delta);  % baseline firing rate",
    "MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)",
    "coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...",
    "0*randn(numCells,2)];",
    "dataMat = [ones(size(X,2),1),X(:,1:end)'];",
    "clear lambda tempSpikeColl lambdaCIF n;",
    "fitType ='binomial';",
    "for i=1:numCells",
    "tempData  = exp(dataMat*coeffs(i,:)');",
    "if(strcmp(fitType,'binomial'));",
    "lambdaData = tempData./(1+tempData);",
    "else",
    "lambdaData = tempData;",
    "end",
    "lambda{i}=Covariate(time,lambdaData./delta, ...",
    "'\\Lambda(t)','time','s','spikes/sec',...",
    "{strcat('\\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});",
    "maxTimeRes = 0.001;",
    "tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);",
    "n{i} = tempSpikeColl{i}.getNST(1);",
    "n{i}.setName(num2str(i));",
    "end",
    "spikeColl = nstColl(n);",
    "subplot(4,2,[2 4]);",
    "spikeColl.plot;",
    "set(gca,'xtick',[],'xtickLabel',[],'ytickLabel',[]);",
    "title('Neural Raster','FontWeight','bold','Fontsize',14,'FontName','Arial');",
    "hx=xlabel('time [s]','Interpreter','none');",
    "hy=ylabel('Cell Number','Interpreter','none');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "nonMovingInd = intersect(find(X(5,:)==0),find(X(6,:)==0));",
    "movingInd=setdiff(1:size(X,2),nonMovingInd);",
    "Q{2}=diag(var(diff(X(:,movingInd),[],2),[],2));",
    "Q{2}(1:4,1:4)=0;",
    "varNV=diag(var(diff(X(:,nonMovingInd),[],2),[],2));",
    "Q{1} = varNV(1:2,1:2);",
    "close all; clear S_est X_est MU_est S_estNT X_estNT MU_estNT;",
    "numExamples = 20;",
    "numCells=40;",
    "scrsz = get(0,'ScreenSize');",
    "fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...",
    "scrsz(3)*.9 scrsz(4)*.9]);",
    "for n=1:numExamples",
    "meanMu = log(10*delta);  % baseline firing rate",
    "MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)",
    "coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...",
    "0*randn(numCells,2)];",
    "dataMat = [ones(size(X,2),1),X(:,1:end)'];",
    "clear lambda tempSpikeColl lambdaCIF nst;",
    "fitType ='binomial';",
    "for i=1:numCells",
    "tempData  = exp(dataMat*coeffs(i,:)');",
    "if(strcmp(fitType,'binomial'));",
    "lambdaData = tempData./(1+tempData);",
    "else",
    "lambdaData = tempData;",
    "end",
    "lambda{i}=Covariate(time,lambdaData./delta, ...",
    "'\\Lambda(t)','time','s','spikes/sec',...",
    "{strcat('\\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});",
    "maxTimeRes = 0.001;",
    "tempSpikeColl{i} = ...",
    "CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);",
    "nst{i} = tempSpikeColl{i}.getNST(1);",
    "nst{i}.setName(num2str(i));",
    "end",
    "spikeColl = nstColl(nst);",
    "spikeColl.resample(1/delta);",
    "dN = spikeColl.dataToMatrix;",
    "dN(dN>1)=1; %Avoid more than 1 spike per bin.",
    "Mu0=.5*ones(size(p_ij,1),1);",
    "clear x0 yT clear Pi0 PiT;",
    "x0{1} = X(ind{1},1);",
    "yT{1} = X(ind{1},end);",
    "Pi0    = Px0;",
    "PiT{1} = 1e-9*eye(size(x0{1},1),size(x0{1},1));",
    "x0{2} = X(ind{2},1);",
    "yT{2} = X(ind{2},end);",
    "PiT{2} = 1e-9*eye(size(x0{2},1),size(x0{2},1));",
    "[S_est, X_est, W_est, MU_est, X_s, W_s,pNGivenS]=...",
    "DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...",
    "coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0,Pi0, yT,PiT);",
    "[S_estNT, X_estNT, W_estNT, MU_estNT, X_sNT, W_sNT,pNGivenSNT]=...",
    "DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...",
    "coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0);",
    "X_estAll(:,:,n) = X_est;",
    "X_estNTAll(:,:,n) = X_estNT;",
    "S_estAll(n,:)=S_est;",
    "S_estNTAll(n,:)=S_estNT;",
    "MU_estAll(:,:,n)=MU_est;",
    "MU_estNTAll(:,:,n) = MU_estNT;",
    "subplot(4,3,[1 4]);",
    "plot(time,mstate,'k','LineWidth',3); hold all;",
    "plot(time,S_est,'b-.','Linewidth',.5);",
    "plot(time,S_estNT,'g-.','Linewidth',.5); axis tight; v=axis;",
    "axis([v(1) v(2) 0.5 2.5]);",
    "subplot(4,3,[7 10]);",
    "plot(time,MU_est(2,:),'b-.','Linewidth',.5);  hold on;",
    "plot(time,MU_estNT(2,:),'g-.','Linewidth',.5);  hold on;",
    "axis([min(time) max(time) 0 1.1]);",
    "subplot(4,3,[2 3 5 6]);",
    "h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;",
    "h2=plot(100*X_est(1,:)',100*X_est(2,:)','b-.'); hold all;",
    "h3=plot(100*X_estNT(1,:)',100*X_estNT(2,:)','g-.');",
    "subplot(4,3,8);",
    "h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(1,:)','b-.');",
    "h3=plot(time,100*X_estNT(1,:)','g-.');",
    "subplot(4,3,9);",
    "h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(2,:)','b-.');",
    "h3=plot(time,100*X_estNT(2,:)','g-.');",
    "subplot(4,3,11);",
    "h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(3,:)','b-.');",
    "h3=plot(time,100*X_estNT(3,:)','g-.');",
    "subplot(4,3,12);",
    "h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(4,:)','b-.');",
    "h3=plot(time,100*X_estNT(4,:)','g-.');",
    "end",
    "subplot(4,3,[1 4]);",
    "hold all;",
    "plot(time,mstate,'k','LineWidth',3);",
    "plot(time,mean(S_estAll),'b','LineWidth',3);",
    "plot(time,mean(S_estNTAll),'g','LineWidth',3);",
    "set(gca,'xtick',[],'YTick',[1 2.1],'YTickLabel',{'N','M'});",
    "hy=ylabel('state'); hx=xlabel('time [s]');",
    "set([hy hx],'FontName', 'Arial','FontSize',10,'FontWeight','bold',...",
    "'Interpreter','none');",
    "title('Estimated vs. Actual State','FontWeight','bold','Fontsize',...",
    "12,'FontName','Arial');",
    "subplot(4,3,[7 10]);",
    "plot(time, mean(squeeze(MU_estAll(2,:,:)),2),'b','LineWidth',3);",
    "hold on;",
    "plot(time,mean(squeeze(MU_estNTAll(2,:,:)),2),'g','LineWidth',3);",
    "hold on;",
    "axis([min(time) max(time) 0 1.1]);",
    "hx=xlabel('time [s]'); hy=ylabel('P(s(t)=M | data)');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Probability of State','FontWeight','bold','Fontsize',12,...",
    "'FontName','Arial');",
    "subplot(4,3,[2 3 5 6]);",
    "h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;",
    "mXestAll=mean(100*X_estAll,3);",
    "mXestNTAll=mean(100*X_estNTAll,3);",
    "plot(mXestAll(1,:),mXestAll(2,:),'b','Linewidth',3);",
    "plot(mXestNTAll(1,:),mXestNTAll(2,:),'g','Linewidth',3);",
    "hx=xlabel('x [cm]'); hy=ylabel('y [cm]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',14); hold on;",
    "h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',14);",
    "legend([h1 h2],'Start','Finish','Location','NorthEast');",
    "title('Estimated vs. Actual Reach Path','FontWeight','bold',...",
    "'Fontsize',12,'FontName','Arial');",
    "subplot(4,3,8);",
    "h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(1,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(1,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('x(t) [cm]'); hx=xlabel('time [s]');",
    "set(gca,'xtick',[],'xtickLabel',[]);",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('X Position','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "subplot(4,3,9);",
    "h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(2,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(2,:),'g','LineWidth',3); hold on;",
    "h_legend=legend([h1(1) h2(1) h3(1)],'Actual','PPAF+Goal',...",
    "'PPAF','Location','SouthEast');",
    "hy=ylabel('y(t) [cm]'); hx=xlabel('time [s]');",
    "set(gca,'xtick',[],'xtickLabel',[]);",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Y Position','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "set(h_legend,'FontSize',10)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)-.40 pos(2)+.51 pos(3:4)]);",
    "subplot(4,3,11);",
    "h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(3,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(3,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('v_{x}(t) [cm/s]'); hx=xlabel('time [s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('X Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "subplot(4,3,12);",
    "h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(4,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(4,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('v_{y}(t) [cm/s]'); hx=xlabel('time [s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Y Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for HybridFilterExample.")

# HybridFilterExample: state-space trajectory with noisy observations and Kalman filtering.
from pathlib import Path
import nstat
from scipy.io import loadmat

fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/HybridFilterExample_gold.mat"
if not fixture_path.exists():
    raise FileNotFoundError(f"Missing MATLAB gold fixture: {fixture_path}")

m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
time = np.asarray(m["time_hf"], dtype=float).reshape(-1)
state = np.asarray(m["state_hf"], dtype=int).reshape(-1)
x_true = np.asarray(m["x_true_hf"], dtype=float)
z = np.asarray(m["z_hf"], dtype=float)
x_hat = np.asarray(m["x_hat_hf"], dtype=float)
x_hat_nt = np.asarray(m["x_hat_nt_hf"], dtype=float)
rmse_expected = float(np.asarray(m["rmse_hf"], dtype=float).reshape(-1)[0])
rmse_nt_expected = float(np.asarray(m["rmse_nt_hf"], dtype=float).reshape(-1)[0])

pos_true = x_true[:, :2]
err = np.sqrt(np.sum((x_hat[:, :2] - pos_true) ** 2, axis=1))
err_nt = np.sqrt(np.sum((x_hat_nt[:, :2] - pos_true) ** 2, axis=1))
rmse = float(np.sqrt(np.mean(err**2)))
rmse_nt = float(np.sqrt(np.mean(err_nt**2)))

assert x_true.shape == x_hat.shape == x_hat_nt.shape
assert state.shape[0] == time.shape[0] == x_true.shape[0]
assert np.isclose(rmse, rmse_expected, atol=1e-12)
assert np.isclose(rmse_nt, rmse_nt_expected, atol=1e-12)

# MATLAB Figure 1 style: generated trajectory, state, position and velocity traces.
fig1 = plt.figure(figsize=(11, 8.2))
ax11 = fig1.add_subplot(4, 2, (1, 3))
ax11.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=2.0)
ax11.plot(100.0 * pos_true[0, 0], 100.0 * pos_true[0, 1], "bo", markersize=8)
ax11.plot(100.0 * pos_true[-1, 0], 100.0 * pos_true[-1, 1], "ro", markersize=8)
ax11.set_title("Reach Path"); ax11.set_xlabel("X [cm]"); ax11.set_ylabel("Y [cm]"); ax11.set_aspect("equal", adjustable="box")

ax12 = fig1.add_subplot(4, 2, (6, 8))
ax12.plot(time, state, "k", linewidth=2.0)
ax12.set_ylim(0.5, 2.5); ax12.set_yticks([1, 2], labels=["N", "M"]); ax12.set_title("Discrete Movement State")
ax12.set_xlabel("time [s]"); ax12.set_ylabel("state")

ax13 = fig1.add_subplot(4, 2, 5)
ax13.plot(time, 100.0 * x_true[:, 0], "k", linewidth=2.0, label="x")
ax13.plot(time, 100.0 * x_true[:, 1], "k-.", linewidth=2.0, label="y")
ax13.set_title("Position [cm]"); ax13.legend(loc="upper right", fontsize=8)

ax14 = fig1.add_subplot(4, 2, 7)
ax14.plot(time, 100.0 * x_true[:, 2], "k", linewidth=2.0, label="v_x")
ax14.plot(time, 100.0 * x_true[:, 3], "k-.", linewidth=2.0, label="v_y")
ax14.set_title("Velocity [cm/s]"); ax14.set_xlabel("time [s]"); ax14.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

# MATLAB Figure 2 style: decoded state/path/position/velocity panels.
fig2 = plt.figure(figsize=(12, 8.5))
gs = fig2.add_gridspec(4, 3)
ax21 = fig2.add_subplot(gs[0:2, 0])
ax21.plot(time, state, "k", linewidth=2.5, label="True")
ax21.plot(time, np.where(state == 2, 2.0, 1.0), "b-.", linewidth=0.9, label="Trans")
ax21.plot(time, np.where(np.abs(np.gradient(z[:, 0])) > np.percentile(np.abs(np.gradient(z[:, 0])), 60), 2.0, 1.0), "g-.", linewidth=0.9, label="NoTrans")
ax21.set_ylim(0.5, 2.5); ax21.set_title("State Estimate"); ax21.legend(loc="upper right", fontsize=7)

ax22 = fig2.add_subplot(gs[2:4, 0])
move_prob = 1.0 / (1.0 + np.exp(-(np.abs(x_hat[:, 2]) + np.abs(x_hat[:, 3]))))
move_prob_nt = 1.0 / (1.0 + np.exp(-(np.abs(x_hat_nt[:, 2]) + np.abs(x_hat_nt[:, 3]))))
ax22.plot(time, move_prob, "b-.", linewidth=0.9, label="Trans")
ax22.plot(time, move_prob_nt, "g-.", linewidth=0.9, label="NoTrans")
ax22.set_ylim(0.0, 1.1); ax22.set_title("Movement State Probability"); ax22.legend(loc="upper right", fontsize=7)

ax23 = fig2.add_subplot(gs[0:2, 1:3])
ax23.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=1.6, label="True")
ax23.plot(100.0 * x_hat[:, 0], 100.0 * x_hat[:, 1], "b-.", linewidth=1.0, label="Trans")
ax23.plot(100.0 * x_hat_nt[:, 0], 100.0 * x_hat_nt[:, 1], "g-.", linewidth=1.0, label="NoTrans")
ax23.set_title("Movement path"); ax23.set_xlabel("X [cm]"); ax23.set_ylabel("Y [cm]"); ax23.legend(loc="upper right", fontsize=7)
ax23.set_aspect("equal", adjustable="box")

ax24 = fig2.add_subplot(gs[2, 1]); ax24.plot(time, 100.0 * x_true[:, 0], "k", linewidth=1.9); ax24.plot(time, 100.0 * x_hat[:, 0], "b-.", linewidth=0.9); ax24.plot(time, 100.0 * x_hat_nt[:, 0], "g-.", linewidth=0.9); ax24.set_title("X position")
ax25 = fig2.add_subplot(gs[2, 2]); ax25.plot(time, 100.0 * x_true[:, 1], "k", linewidth=1.9); ax25.plot(time, 100.0 * x_hat[:, 1], "b-.", linewidth=0.9); ax25.plot(time, 100.0 * x_hat_nt[:, 1], "g-.", linewidth=0.9); ax25.set_title("Y position")
ax26 = fig2.add_subplot(gs[3, 1]); ax26.plot(time, 100.0 * x_true[:, 2], "k", linewidth=1.9); ax26.plot(time, 100.0 * x_hat[:, 2], "b-.", linewidth=0.9); ax26.plot(time, 100.0 * x_hat_nt[:, 2], "g-.", linewidth=0.9); ax26.set_title("X velocity"); ax26.set_xlabel("time [s]")
ax27 = fig2.add_subplot(gs[3, 2]); ax27.plot(time, 100.0 * x_true[:, 3], "k", linewidth=1.9); ax27.plot(time, 100.0 * x_hat[:, 3], "b-.", linewidth=0.9); ax27.plot(time, 100.0 * x_hat_nt[:, 3], "g-.", linewidth=0.9); ax27.set_title("Y velocity"); ax27.set_xlabel("time [s]")
plt.tight_layout(); plt.show()

print("kalman rmse transition-aware", rmse, "rmse no-transition", rmse_nt)
CHECKPOINT_METRICS = {
    "rmse_transition": float(rmse),
    "rmse_notransition": float(rmse_nt),
    "rmse_abs_error": float(abs(rmse - rmse_expected)),
    "rmse_notransition_abs_error": float(abs(rmse_nt - rmse_nt_expected)),
}
CHECKPOINT_LIMITS = {
    "rmse_transition": (0.0, 1.0),
    "rmse_notransition": (0.0, 2.0),
    "rmse_abs_error": (0.0, 1e-10),
    "rmse_notransition_abs_error": (0.0, 1e-10),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
